# Text2Cypher Generation with Claude 3.5 Haiku via AWS Bedrock

This notebook generates Cypher queries from natural language questions using Claude 3.5 Haiku through AWS Bedrock.

## Setup
- Make sure AWS credentials are configured
- Ensure you have access to Claude 3.5 Haiku in us-east-1
- Budget-friendly approach: Default to 10 questions 

In [1]:
# Install required packages
!pip install --quiet boto3 neo4j pandas

In [2]:
import pandas as pd
import boto3
import json
import time
from typing import Dict, Any
import os

In [3]:
# Load the questions dataset
questions = pd.read_csv("text2cypher_questions.csv")
print(f"Total questions available: {len(questions)}")
questions.head()

Total questions available: 9824


,question,type,database
0,What are the top 5 movies with a runtime great...,Simple Retrieval Queries,recommendations
1,List the first 3 directors born before 1950.,Simple Retrieval Queries,recommendations
2,Which 5 users have rated more than 20 movies?,Simple Retrieval Queries,recommendations
3,Identify the top 5 actors who have acted in mo...,Simple Retrieval Queries,recommendations
4,What are the top 3 genres associated with movi...,Simple Retrieval Queries,recommendations


In [5]:
# SAMPLING CONFIGURATION - Modify this cell to change sample size
# Uncomment and modify the line below to change sample size:

SAMPLE_SIZE = 10  
# SAMPLE_SIZE = 2000  # Uncomment for larger sample
# SAMPLE_SIZE = len(questions)  # Uncomment to process all questions (~$5-10)

# Select questions from supported databases
selected_questions = questions[
    questions["database"].isin([
        "recommendations",
        "companies",
        "twitch",
        "twitter",
        "gameofthrones",
        "movies",
        "neoflix",
    ])
].sample(n=min(SAMPLE_SIZE, len(questions)), random_state=42).reset_index(drop=True)

print(f"Processing {len(selected_questions)} questions")
print(f"Estimated cost with Haiku 3.5: $0.{len(selected_questions)//200}-{len(selected_questions)//100}")
selected_questions.head()

Processing 10 questions
Estimated cost with Haiku 3.5: $0.0-0


,question,type,database
0,Name the top 3 organizations by revenue within...,Complex Aggregation Queries,companies
1,List the first 3 tweets retweeted by users loc...,Simple Retrieval Queries,twitter
2,What are the original titles of movies that ha...,Complex Retrieval Queries,neoflix
3,Display the top 5 movies with the highest budg...,Complex Retrieval Queries,neoflix
4,List the industry categories of organizations ...,Complex Retrieval Queries,companies


In [6]:
# Load schemas
schemas = pd.read_csv('text2cypher_schemas.csv')
schemas.head()

# Create schema dictionary
schema_dict = {}
for i, row in schemas.iterrows():
    schema_dict[row['database']] = row['schema']
    
print(f"Loaded schemas for {len(schema_dict)} databases")

Loaded schemas for 15 databases


In [7]:
# AWS Bedrock setup for Claude 3.5 Haiku
import boto3

# Initialize Bedrock Runtime client
bedrock_runtime = boto3.client(
    service_name='bedrock-runtime',
    region_name='us-east-1'  # or us-west-2
)

# Claude 3.5 Haiku inference profile ID
MODEL_ID = "us.anthropic.claude-3-5-haiku-20241022-v1:0"

print(f"Using model: {MODEL_ID}")
print(f"Region: us-east-1")

Using model: us.anthropic.claude-3-5-haiku-20241022-v1:0
Region: us-east-1


In [8]:
# Define the Cypher generation function
def generate_cypher_with_haiku(question: str, schema: str) -> str:
    """
    Generate Cypher query using Claude 3.5 Haiku via AWS Bedrock
    """
    # System prompt following neo4j-labs methodology
    system_prompt = """Given an input question, convert it to a Cypher query. No pre-amble.
Additional instructions:
- Ensure that queries checking for non-null properties use `IS NOT NULL` in a straightforward manner.
- Don't use `size((n)--(m))` for counting relationships. Instead use the new `count{(n)--(m)}` syntax.
- Incorporate the new existential subqueries in examples where the query needs to check for the existence of a pattern.
  Example: MATCH (p:Person)-[r:IS_FRIENDS_WITH]->(friend:Person)
            WHERE exists{ (p)-[:WORKS_FOR]->(:Company {name: 'Neo4j'})}
            RETURN p, r, friend"""
    
    # User prompt template
    user_prompt = f"""Based on the Neo4j graph schema below, write a Cypher query that would answer the user's question:
{schema}

Question: {question}
Cypher query:"""
    
    # Prepare the request payload
    payload = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 4000,
        "temperature": 0.0,
        "messages": [
            {
                "role": "user",
                "content": f"{system_prompt}\n\n{user_prompt}"
            }
        ]
    }
    
    try:
        # Make the API call
        response = bedrock_runtime.invoke_model(
            modelId=MODEL_ID,
            body=json.dumps(payload)
        )
        
        # Parse the response
        response_body = json.loads(response['body'].read())
        
        # Extract the generated text
        cypher = response_body['content'][0]['text'].strip()
        return cypher
        
    except Exception as e:
        print(f"Error generating Cypher: {e}")
        return ""

# Test the function with a sample
if len(selected_questions) > 0:
    test_row = selected_questions.iloc[0]
    test_schema = schema_dict[test_row['database']]
    test_cypher = generate_cypher_with_haiku(test_row['question'], test_schema)
    print("Test generation:")
    print(f"Question: {test_row['question']}")
    print(f"Generated Cypher: {test_cypher}")

Test generation:
Question: Name the top 3 organizations by revenue within the 'Financial Services' category.
Generated Cypher: MATCH (o:Organization)-[:HAS_CATEGORY]->(ic:IndustryCategory {name: 'Financial Services'})
WHERE o.revenue IS NOT NULL
RETURN o.name, o.revenue
ORDER BY o.revenue DESC
LIMIT 3


In [9]:
# Generate Cypher queries for all selected questions
print(f"Starting generation for {len(selected_questions)} questions...")
print("This may take a while. Progress will be shown every question.")

cypher_responses = []
start_time = time.time()

for i, row in selected_questions.iterrows():
    if i % 1 == 0:   ## change this for bigger batch size and sample size
        elapsed = time.time() - start_time
        print(f"Progress: {i}/{len(selected_questions)} questions processed. Elapsed: {elapsed:.1f}s")
    
    schema = schema_dict[row['database']]
    
    try:
        cypher = generate_cypher_with_haiku(row['question'], schema)
        cypher_responses.append({
            "question": row["question"], 
            "database": row["database"], 
            "cypher": cypher, 
            "type": row["type"]
        })
    except Exception as e:
        print(f"Error at question {i}: {e}")
        cypher_responses.append({
            "question": row["question"], 
            "database": row["database"], 
            "cypher": "", 
            "type": row["type"]
        })

total_time = time.time() - start_time
print(f"\nGeneration completed!")
print(f"Total time: {total_time:.1f} seconds")
print(f"Average time per question: {total_time/len(selected_questions):.2f} seconds")
print(f"Generated {len(cypher_responses)} responses")

Starting generation for 10 questions...
This may take a while. Progress will be shown every question.
Progress: 0/10 questions processed. Elapsed: 0.0s
Progress: 1/10 questions processed. Elapsed: 1.9s
Progress: 2/10 questions processed. Elapsed: 3.8s
Progress: 3/10 questions processed. Elapsed: 5.8s
Progress: 4/10 questions processed. Elapsed: 8.3s
Progress: 5/10 questions processed. Elapsed: 10.3s
Progress: 6/10 questions processed. Elapsed: 12.1s
Progress: 7/10 questions processed. Elapsed: 13.9s
Progress: 8/10 questions processed. Elapsed: 15.6s
Progress: 9/10 questions processed. Elapsed: 17.7s

Generation completed!
Total time: 19.7 seconds
Average time per question: 1.97 seconds
Generated 10 responses


In [10]:
# Create DataFrame from responses
combined_df = pd.DataFrame(cypher_responses)
print(f"Created DataFrame with {len(combined_df)} rows")
combined_df.head()

Created DataFrame with 10 rows


,question,database,cypher,type
0,Name the top 3 organizations by revenue within...,companies,MATCH (o:Organization)-[:HAS_CATEGORY]->(ic:In...,Complex Aggregation Queries
1,List the first 3 tweets retweeted by users loc...,twitter,MATCH (u:User {location: 'Graphs Are Everywher...,Simple Retrieval Queries
2,What are the original titles of movies that ha...,neoflix,MATCH (m:Movie)-[:SPOKEN_IN_LANGUAGE]->(l:Lang...,Complex Retrieval Queries
3,Display the top 5 movies with the highest budg...,neoflix,MATCH (m:Movie)\nWHERE m.budget > 0 AND m.reve...,Complex Retrieval Queries
4,List the industry categories of organizations ...,companies,MATCH (o:Organization)-[:HAS_CEO]->(p:Person)\...,Complex Retrieval Queries


In [17]:
# Validation against Neo4j demo databases
from neo4j import GraphDatabase

DEMO_URL = "neo4j+s://demo.neo4jlabs.com"

print("Starting validation against Neo4j demo databases...")
print("This will test each query for syntax errors, timeouts, and result availability.")

syntax_error = []
returns_results = []
timeouts = []
false_schema = []  # Will be computed later

last_graph = ""
driver = None

for i, row in combined_df.reset_index().iterrows():
    if i % 1 == 0: ## change this for bigger batch size and sample size
        print(f"Validation progress: {i}/{len(combined_df)}")
    
    # Connect to new database if needed
    if row['database'] != last_graph:
        if driver:
            driver.close()
        last_graph = row["database"]
        print(f"Connecting to database: {last_graph}")
        driver = GraphDatabase.driver(
            DEMO_URL, 
            auth=(row["database"], row["database"])
        )
    
    # Skip empty queries
    if not row['cypher'].strip():
        syntax_error.append(True)
        returns_results.append(False)
        timeouts.append(False)
        continue
    
    try:
        with driver.session(database=row["database"]) as session:
            # Set timeout to 10 seconds
            result = session.run(row['cypher'], timeout=10)
            data = result.data()
            
            if data:
                returns_results.append(True)
            else:
                returns_results.append(False)
            syntax_error.append(False)
            timeouts.append(False)
            
    except Exception as e:
        error_str = str(e).lower()
        
        if any(keyword in error_str for keyword in ['syntax', 'invalid', 'parse']):
            syntax_error.append(True)
            print(f"Syntax error in query {i}: {e}")
        else:
            syntax_error.append(False)
        
        if 'timeout' in error_str or 'time' in error_str:
            timeouts.append(True)
        else:
            timeouts.append(False)
            
        returns_results.append(False)

# Close the last driver
if driver:
    driver.close()

print("\nValidation completed!")

Starting validation against Neo4j demo databases...
This will test each query for syntax errors, timeouts, and result availability.
Validation progress: 0/10
Connecting to database: companies
Validation progress: 1/10
Connecting to database: twitter
Validation progress: 2/10
Connecting to database: neoflix
Validation progress: 3/10
Validation progress: 4/10
Connecting to database: companies
Validation progress: 5/10
Connecting to database: neoflix
Validation progress: 6/10
Connecting to database: recommendations
Validation progress: 7/10
Connecting to database: twitter
Validation progress: 8/10
Connecting to database: neoflix
Validation progress: 9/10

Validation completed!


In [18]:
# Add validation results to DataFrame
combined_df["syntax_error"] = syntax_error
combined_df["timeout"] = timeouts
combined_df["returns_results"] = returns_results

# Implement false_schema detection
print("\nDetecting schema hallucinations...")
false_schema = []

import re

for i, row in combined_df.iterrows():
    schema = schema_dict[row['database']]
    cypher = row['cypher']
    
    # Basic schema hallucination detection
    # Extract node labels from schema (e.g., :Movie, :Person)
    schema_labels = set(re.findall(r':(\w+)', schema))
    
    # Extract node labels used in cypher
    cypher_labels = set(re.findall(r':(\w+)', cypher))
    
    # Extract relationship types from schema (e.g., -[:ACTED_IN]->)
    schema_rels = set(re.findall(r'\[:([A-Z_]+)\]', schema))
    
    # Extract relationship types used in cypher
    cypher_rels = set(re.findall(r'\[:([A-Z_]+)\]', cypher))
    
    # Check if cypher uses labels/relationships not in schema
    invalid_labels = cypher_labels - schema_labels
    invalid_rels = cypher_rels - schema_rels
    
    if invalid_labels or invalid_rels:
        false_schema.append(True)
        if i < 5:  # Show first few examples
            if invalid_labels:
                print(f"Invalid labels in query {i}: {invalid_labels}")
            if invalid_rels:
                print(f"Invalid relationships in query {i}: {invalid_rels}")
    else:
        false_schema.append(False)

combined_df["false_schema"] = false_schema

print("Validation results summary:")
print(f"Syntax errors: {sum(syntax_error)} / {len(syntax_error)} ({100*sum(syntax_error)/len(syntax_error):.1f}%)")
print(f"Timeouts: {sum(timeouts)} / {len(timeouts)} ({100*sum(timeouts)/len(timeouts):.1f}%)")
print(f"Returns results: {sum(returns_results)} / {len(returns_results)} ({100*sum(returns_results)/len(returns_results):.1f}%)")
print(f"Schema hallucinations: {sum(false_schema)} / {len(false_schema)} ({100*sum(false_schema)/len(false_schema):.1f}%)")


Detecting schema hallucinations...
Validation results summary:
Syntax errors: 0 / 10 (0.0%)
Timeouts: 0 / 10 (0.0%)
Returns results: 9 / 10 (90.0%)
Schema hallucinations: 0 / 10 (0.0%)


In [19]:
# Create final dataset with proper column order
final_df = combined_df[[
    "question",
    "cypher",
    "type",
    "database",
    "syntax_error",
    "timeout",
    "returns_results",
    "false_schema"
]]

print(f"Final dataset shape: {final_df.shape}")
final_df.head()

Final dataset shape: (10, 8)


,question,cypher,type,database,syntax_error,timeout,returns_results,false_schema
0,Name the top 3 organizations by revenue within...,MATCH (o:Organization)-[:HAS_CATEGORY]->(ic:In...,Complex Aggregation Queries,companies,False,False,False,False
1,List the first 3 tweets retweeted by users loc...,MATCH (u:User {location: 'Graphs Are Everywher...,Simple Retrieval Queries,twitter,False,False,True,False
2,What are the original titles of movies that ha...,MATCH (m:Movie)-[:SPOKEN_IN_LANGUAGE]->(l:Lang...,Complex Retrieval Queries,neoflix,False,False,True,False
3,Display the top 5 movies with the highest budg...,MATCH (m:Movie)\nWHERE m.budget > 0 AND m.reve...,Complex Retrieval Queries,neoflix,False,False,True,False
4,List the industry categories of organizations ...,MATCH (o:Organization)-[:HAS_CEO]->(p:Person)\...,Complex Retrieval Queries,companies,False,False,True,False


In [20]:
# Show detailed statistics
print("Detailed Statistics:")
print("\nSyntax Errors:")
print(final_df["syntax_error"].value_counts())
print("\nTimeouts:")
print(final_df["timeout"].value_counts())
print("\nReturns Results:")
print(final_df["returns_results"].value_counts())
print("\nBy Database:")
print(final_df["database"].value_counts())
print("\nBy Question Type:")
print(final_df["type"].value_counts())

Detailed Statistics:

Syntax Errors:
syntax_error
False    10
Name: count, dtype: int64

Timeouts:
timeout
False    10
Name: count, dtype: int64

Returns Results:
returns_results
True     9
False    1
Name: count, dtype: int64

By Database:
database
neoflix            5
companies          2
twitter            2
recommendations    1
Name: count, dtype: int64

By Question Type:
type
Complex Retrieval Queries      3
Pathfinding Queries            3
Complex Aggregation Queries    2
Simple Retrieval Queries       2
Name: count, dtype: int64


In [21]:
# Save the final dataset
output_filename = "text2cypher_haiku35.csv"
final_df.to_csv(output_filename, index=False)
print(f"Dataset saved as: {output_filename}")
print(f"Total questions processed: {len(final_df)}")

# Show sample of successful queries
successful_queries = final_df[
    (~final_df["syntax_error"]) & 
    (~final_df["timeout"]) & 
    (final_df["returns_results"])
]
print(f"\nSuccessful queries: {len(successful_queries)} / {len(final_df)} ({100*len(successful_queries)/len(final_df):.1f}%)")

if len(successful_queries) > 0:
    print("\nSample successful query:")
    sample = successful_queries.iloc[0]
    print(f"Question: {sample['question']}")
    print(f"Database: {sample['database']}")
    print(f"Cypher: {sample['cypher']}")

Dataset saved as: text2cypher_haiku35.csv
Total questions processed: 10

Successful queries: 9 / 10 (90.0%)

Sample successful query:
Question: List the first 3 tweets retweeted by users located in 'Graphs Are Everywhere'.
Database: twitter
Cypher: MATCH (u:User {location: 'Graphs Are Everywhere'})-[:POSTS]->(t1:Tweet)<-[:RETWEETS]-(t2:Tweet)
RETURN DISTINCT t2
LIMIT 3


## Summary

This notebook generates a synthetic text2cypher dataset using Claude 3.5 Haiku via AWS Bedrock.

### Key Features:
- **Budget-friendly**: Default 10 questions 
- **Scalable**: Easy to modify sample size
- **Validated**: Tests against live Neo4j databases
- **Compatible**: Follows neo4j-labs methodology

### Usage:
1. Configure AWS credentials for Bedrock access
2. Modify `SAMPLE_SIZE` in the sampling cell as needed
3. Run all cells to generate and validate the dataset
4. Output saved as `text2cypher_haiku35.csv`

### Cost Estimation:
- 1,000 questions: ~$0.50-1.00
- 5,000 questions: ~$2.50-5.00
- All questions (~9,824): ~$5-10

Claude 3.5 Haiku provides an excellent balance of cost-effectiveness and performance for large-scale text2cypher dataset generation.